# Tema 5: Procesadores avanzados (AIC)

> **Material de estudio** basado en `Procesadores avanzados 1a parte.pdf` y `Procesadores avanzados 1a y 2a parte.pdf` (Salvador Vicente Petit Martí — *Arquitectura e Ingeniería de Computadores para IA*, Universitat Politècnica de València [UPV], curso 2026/27). *(Nota: no existe un boletín de actividades independiente; la Parte II recoge y resuelve paso a paso todas las cuestiones y ejercicios propuestos en las transparencias de ambos PDF.)*

Este documento reúne y explica de forma autosuficiente toda la teoría del **Tema 5 (Procesadores avanzados)** desarrollada en las transparencias —operaciones multiciclo y ejecución fuera de orden con especulación, ROB, *I Queue* y *LSQ*—, junto con sus diagramas, tablas comparativas, aclaraciones de notación y la resolución paso a paso de todos los ejercicios planteados en el material.

---

## Tabla de contenidos

- [Parte I: Explicación completa de la teoría](#parte-i-explicación-completa-de-la-teoría)
  - [1. Introducción](#1-introducción)
  - [2. Operaciones multiciclo](#2-operaciones-multiciclo)
  - [3. Ejecución fuera de orden](#3-ejecución-fuera-de-orden)
- [Parte II: Resolución paso a paso de las actividades](#parte-ii-resolución-paso-a-paso-de-las-actividades)
  - [Actividades 1: Cuestiones y ejercicios de operaciones multiciclo](#actividades-1-cuestiones-y-ejercicios-de-operaciones-multiciclo)
  - [Actividades 2: Ejercicios de ejecución fuera de orden (o-o-o)](#actividades-2-ejercicios-de-ejecución-fuera-de-orden-o-o-o)

---

# Parte I: Explicación completa de la teoría

## 1. Introducción

En un procesador segmentado básico de 5 etapas (`IF`, `ID`, `EX`, `ME`, `WB`), todas las operaciones aritméticas y lógicas se completan en un único ciclo dentro de la etapa `EX`, y todos los accesos a memoria en un único ciclo dentro de `ME`. Sin embargo, los procesadores reales deben ejecutar instrucciones mucho más costosas —como sumas, multiplicaciones y divisiones en coma flotante, o accesos a caché con latencia superior a un ciclo— y, al mismo tiempo, evitar que la dependencia de una instrucción lenta paralice al resto del procesador.

El **Tema 5** aborda precisamente las técnicas microarquitectónicas que permiten dar ese salto hacia procesadores de altas prestaciones. En el índice general de las transparencias (transparencias 2 y 40) el tema se estructura en cinco bloques:

1. **Operaciones multiciclo:** cómo adaptar el cauce segmentado (*pipeline*) para soportar operadores cuya fase de ejecución dura varios ciclos sin degradar la frecuencia de reloj, junto con sus nuevos cortocircuitos y riesgos.
2. **Ejecución fuera de orden:** cómo eliminar el cuello de botella de la ejecución estrictamente en orden mediante la microarquitectura con *Dispatch*, *I Queue*, *Issue*, *Reorder Buffer* (*ROB*), *Common Data Bus* (*CDB*), *Commit* y gestión de *loads/stores* con *LSQ*.
3. **Predicción dinámica** *(solo enunciada en el índice general)*.
4. **Procesadores superescalares** *(solo enunciada en el índice general)*.
5. **Procesadores multihilo** *(solo enunciada en el índice general)*.

*(Nota sobre la cobertura de los PDF: tanto `Procesadores avanzados 1a parte.pdf` como `Procesadores avanzados 1a y 2a parte.pdf` desarrollan íntegramente los bloques **1 (Operaciones multiciclo)** y **2 (Ejecución fuera de orden)**. Los bloques 3, 4 y 5 aparecen únicamente listados en el índice de las transparencias 2 y 40, sin diapositivas de desarrollo en estos archivos.)*

---

## 2. Operaciones multiciclo

### 2.1. Por qué son necesarias las operaciones multiciclo

**Intuición:** la etapa `EX` (*Execute*) del procesador segmentado básico está dimensionada para operaciones sencillas con enteros (sumas, restas, desplazamientos, operaciones lógicas bit a bit), que pueden completarse cómodamente en **1 ciclo de reloj**. En cambio, cuando introducimos operaciones en **coma flotante (c.f.)** o accesos a memoria más lentos, ese tiempo deja de ser suficiente (transparencia 3).

Por ejemplo, para realizar una simple **suma de dos números en coma flotante** el circuito debe encadenar varios pasos internos:

1. Comparar los exponentes de ambos operandos.
2. Desplazar la mantisa del número menor para alinear los exponentes.
3. Sumar las mantisas.
4. Redondear el resultado.
5. Volver a normalizar el número en coma flotante.

Los operadores de **multiplicación** y **división** en coma flotante son todavía más complejos y requieren aún más tiempo. Surge así la pregunta central de diseño: *«¿Cómo segmentamos el pipeline para soportar estas operaciones?»*

---

### 2.2. Alternativas de diseño: alargar el reloj frente a operaciones multiciclo

Ante un operador lento en la etapa `EX`, hay dos enfoques posibles (transparencia 4):

| Alternativa | Descripción intuitiva | Impacto sobre la frecuencia | Viabilidad |
| :--- | :--- | :--- | :--- |
| **1. Aumentar el periodo de reloj** | Hacer que el ciclo de reloj $T_{\text{reloj}}$ sea tan largo como la operación más lenta | Si un multiplicador tarda $5\text{ ns}$ y el resto de etapas $1\text{ ns}$ ($f = 1\text{ GHz}$), el reloj pasaría a $T_{\text{reloj}} = 5\text{ ns} \implies f = 200\text{ MHz}$ | **Descartada:** penaliza a todas las instrucciones rápidas por culpa de las más lentas |
| **2. Operaciones multiciclo** | Mantener fijo el periodo de reloj $T_{\text{reloj}}$ y permitir que `EX` dure múltiples ciclos según la instrucción | El periodo no cambia ($1\text{ ns}$). Una multiplicación de $4\text{ ns}$ ocupa 4 ciclos en ejecución | **Adoptada:** solo las instrucciones complejas tardan más ciclos |

Con la solución multiciclo, una multiplicación de $4\text{ ns}$ en un procesador con ciclo de reloj de $1\text{ ns}$ atraviesa la secuencia de etapas:

```text
IF  ID  EX  EX  EX  EX  ME  WB
```

donde el periodo de reloj $T_{\text{reloj}} = 1\text{ ns}$ permanece intacto y únicamente se prolonga la fase de ejecución durante tantos ciclos como requiera el operador.

---

### 2.3. Tipos de operadores multiciclo: latencia e *Initiation Rate*

Un operador multiciclo puede construirse de dos maneras (transparencia 5), según incorpore o no registros de segmentación internos entre sus subetapas:

- **Operador segmentado (*pipelined*):** el circuito de ejecución se divide en etapas consecutivas de $1$ ciclo (por ejemplo, `M1`, `M2`, `M3`, `M4` para un multiplicador de $4\text{ ns}$ a $1\text{ GHz}$) separadas por **registros de segmentación internos** (`M1/M2`, `M2/M3`, `M3/M4`, además del registro de entrada tras `ID` y el de salida hacia `WB`). Así, en cuanto la primera multiplicación avanza a `M2` en el ciclo 2, una segunda multiplicación puede entrar en `M1` sin esperar a que termine la primera.
- **Operador no segmentado (*unpipelined*):** todo el circuito forma un único bloque combinacional de varios ciclos sin registros intermedios. Mientras una instrucción lo está usando (durante 4 ciclos en el ejemplo), ninguna otra instrucción puede comenzar en ese operador.

*(Nota sobre las transparencias: en la transparencia 5, el segundo epígrafe rojo repite por errata el título «Segmentado» en lugar de «No segmentado», aunque la pregunta inferior de $\text{IR}$ deja claro que se refiere al caso no segmentado).*

Para caracterizar cualquier operador multiciclo se emplean dos métricas fundamentales:

- **Latencia ($L$):** tiempo (medido en nanosegundos o en número de ciclos de reloj) que emplea el circuito desde que entra una operación hasta que produce su resultado.
  *Pregunta intuitiva:* *«¿cuántos ciclos tarda una sola instrucción en atravesar este operador?»*
- ***Initiation Rate* ($\text{IR}$):** inversa del número de ciclos $k$ que deben transcurrir entre el inicio de una operación y el momento en que el circuito puede iniciar una nueva operación:
  $$\text{IR} = \frac{1}{k}$$
  donde:
  - $k \in \mathbb{Z}^+$ es el intervalo de iniciación (*initiation interval*), es decir, el número de ciclos que hay que esperar para lanzar otra operación al mismo operador.
  - Si el operador está **completamente segmentado**, puede aceptar una nueva instrucción cada ciclo ($k = 1$), luego $\text{IR} = 1/1 = 1$.
  - Si el operador es **no segmentado** con latencia de $L$ ciclos, queda ocupado los $L$ ciclos completos ($k = L$), luego $\text{IR} = 1/L$ (en el ejemplo de 4 ciclos, $\text{IR} = 1/4$).

*(Mini ejemplo numérico: en un divisor no segmentado de $6$ ciclos, su latencia es $L = 6$ ciclos y su $\text{IR} = 1/6$; si una división entra al divisor en el ciclo $3$, lo ocupa en los ciclos $3, 4, 5, 6, 7, 8$ y la siguiente división no podrá entrar en `D1` hasta el ciclo $3 + 6 = 9$.)*

---

### 2.4. Organización del *pipeline* con operadores especializados

Al incorporar instrucciones de coma flotante y accesos a caché multiciclo, se añaden **operadores especializados en paralelo** después de la etapa `ID` (transparencias 6 y 7):

1. **Unidad especializada de *load* y *store* (transparencia 6):** el tiempo de acceso a la caché de datos puede ser mayor que $1$ ciclo, por lo que se introduce una unidad específica con etapas de carga (`L1`, `L2`, ...) y de almacenamiento (`S1`, `S2`, ...). Esta unidad **sustituye a la etapa `ME` tradicional**, que ya no es necesaria en el camino de las instrucciones aritméticas o de salto.
2. **Múltiples operadores aritméticos especializados (transparencia 7):** tras decodificar en `ID`, cada instrucción se dirige al operador correspondiente antes de converger en la etapa común de escritura (`WB`):
   - **ALU de enteros multifunción:** `EX` (1 ciclo) $\to$ `WB`.
   - **Sumador de coma flotante:** `A1`, `A2`, ... $\to$ `WB`.
   - **Multiplicador de coma flotante:** `M1`, `M2`, `M3`, ... $\to$ `WB`.
   - **Divisor de coma flotante:** `D1`, `D2`, ... $\to$ `WB`.

```text
                +--> [ L1 ][ L2 ] / [ S1 ][ S2 ] --+
                |                                  |
                +--> [ EX ] (ALU enteros) ---------+
                |                                  |
[ IF ][ ID ] ---+--> [ A1 ][ A2 ] (Sumador c.f.) --+--> [ WB ]
                |                                  |
                +--> [ M1 ][ M2 ][ M3 ] (Mult.) ---+
                |                                  |
                +--> [ D1   D2  ... ] (Divisor) ---+
```

*(Descripción visual de la transparencia 7: las etapas iniciales `IF` e `ID` siguen siendo únicas y compartidas; a la salida de `ID` el flujo se bifurca hacia cinco ramas paralelas —unidad de carga/almacenamiento, ALU de enteros de 1 ciclo, sumador c.f., multiplicador c.f. y divisor c.f.— y todas las salidas confluyen de nuevo hacia la etapa `WB`.)*

---

### 2.5. Cortocircuitos y riesgos en *pipelines* multiciclo

La coexistencia de múltiples operadores con distintas latencias introduce nuevos caminos de cortocircuito y nuevos tipos de riesgos (transparencias 8–11):

#### A. Nuevos cortocircuitos (*forwarding*)

Ahora el dato producido por una instrucción puede necesitarse en la primera etapa de **cualquier otro operador**. Por ello se añaden caminos de cortocircuito *«desde donde se encuentra el dato hasta la etapa en que se necesita»* (transparencia 8), por ejemplo:
- De `A2` a `M1` (el resultado de una suma c.f. pasa directamente a una multiplicación).
- De `L2` a `M1` (el dato recién cargado de caché pasa directamente a una multiplicación).
- De `WB` (o del registro de salida del operador) a `A1`, `M1`, etc.

##### Ejemplo de la teoría: aplicación de cortocircuito (transparencia 9)

Se dispone de un multiplicador c.f. de doble precisión de latencia $4$ segmentado ($\text{IR}=1$) y un sumador c.f. de doble precisión de latencia $2$ segmentado ($\text{IR}=1$). Considérese la secuencia:

```text
                  1   2   3   4   5   6   7   8   9
fmul.d t0,t0,t1  IF  ID  M1  M2  M3  M4  WB
fadd.d t2,t2,t0      IF  id  id  id  ID  A1  A2  WB
fadd.d t3,t3,t4          if  if  if  IF  ID  A1  A2 ...
```

*(Nota sobre la transparencia 9: en la diapositiva original aparecen dos erratas tipográficas: utiliza los nombres `t0..t4` —propios de enteros en RISC-V— en lugar de `f0..f4`, y en la tercera fila escribe `M1 M2 ...` en los ciclos 8 y 9 cuando la instrucción es `fadd.d`, que usa el sumador `A1 A2`. En el diagrama superior se ha corregido `A1 A2` manteniendo el resto exacto de la diapositiva).*

**Qué ocurre ciclo a ciclo:**
- `fmul.d` calcula el nuevo valor de `t0` al terminar `M4` (final del ciclo $6$) y entra en `WB` en el **ciclo $7$**.
- `fadd.d t2,t2,t0` tiene una dependencia verdadera (**RAW**, *Read After Write*) sobre `t0`. Entra en `IF` en el ciclo $2$, pero en los ciclos $3, 4$ y $5$ queda detenida en `ID` (`id id id`) esperando a que el dato esté disponible. En el ciclo $6$ realiza su `ID` efectivo y en el **ciclo $7$** entra en `A1` recibiendo `t0` mediante el **cortocircuito desde `WB` hasta `A1`**.
- Como el cauce es **en orden**, mientras `fadd.d t2` bloquea la etapa `ID` (ciclos $3, 4, 5$), la tercera instrucción (`fadd.d t3,t3,t4`), pese a ser totalmente independiente, no puede avanzar más allá de `IF` (`if if if`).

Como concluye la transparencia 9: *«los cortocircuitos no pueden ocultar el alto coste de las operaciones multiciclo»*.

---

#### B. Riesgos estructurales por operadores no segmentados ($\text{IR} < 1$)

Si un operador no está segmentado ($\text{IR} < 1$) y ya está ocupado por una instrucción anterior, cuando llega una segunda instrucción que necesita ese mismo operador **espera en `ID`** hasta que la instrucción que lo ocupa lo libere (transparencia 8).

##### Ejemplo de la teoría: riesgo estructural en un divisor de 6 ciclos (transparencia 10)

Comparemos el comportamiento de un divisor de $6$ ciclos cuando es segmentado frente a cuando no lo es:

**Caso 1 — Divisor segmentado ($\text{IR} = 1$):** no hay riesgo estructural; la segunda división entra en `D1` en el ciclo siguiente a la primera.

```text
div.f t0,t0,t1  IF  ID  D1  D2  D3  D4  D5  D6  WB
div   t2,t2,t3      IF  ID  D1  D2  D3  D4  D5  D6  WB
add   t4,t4,t5          IF  ID  EX  WB
```

**Caso 2 — Divisor no segmentado ($\text{IR} = 1/6$):**

```text
                  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15
div.f t0,t0,t1   IF  ID  D1  D2  D3  D4  D5  D6  WB
div.f t2,t2,t3       IF  id  id  id  id  id  ID  D1  D2  D3  D4  D5  D6  WB
add   t4,t4,t5           if  if  if  if  if  IF  ID  EX  WB
```

La primera división ocupa el único divisor disponible durante los ciclos $3$ a $8$. La segunda división queda retenida en `ID` durante $5$ ciclos (`id id id id id`) hasta el ciclo $8$, entrando en `D1` en el ciclo $9$. A su vez, la instrucción independiente `add t4,t4,t5` sufre $5$ ciclos de parada en `IF` (`if if if if if`), introduciendo muchos más ciclos de parada que en la ruta de datos básica.

---

#### C. Riesgos estructurales por escritura en el banco de registros

Como las instrucciones tienen duraciones distintas en ejecución, **dos instrucciones pueden llegar al mismo tiempo a `WB`** (transparencia 11). Por ejemplo, una instrucción larga de 4 ciclos que empezó antes puede coincidir en `WB` con una instrucción más joven de 1 o 2 ciclos.

- **Primera medida (hardware):** dividir los registros en **2 bancos de registros independientes**, uno para enteros y otro para reales (coma flotante), tal como se hace en todos los procesadores actuales. Así, una instrucción entera y otra de coma flotante pueden hacer `WB` en el mismo ciclo sin conflicto.
- **Segunda medida (parada):** si aun teniendo dos bancos, en un ciclo dado dos instrucciones quieren escribir en el **mismo** banco de registros, **la instrucción más joven retrasa su escritura en 1 ciclo esperando en `ID`** (al entrar 1 ciclo más tarde al operador, también llegará 1 ciclo más tarde a `WB`).

#### D. Riesgos WAW (*Write After Write*)

En la ruta de datos básica de 5 etapas, todas las instrucciones llegaban a `WB` en orden, por lo que nunca había riesgos WAW. Con operadores multiciclo de distinta latencia, una instrucción más joven y rápida podría alcanzar `WB` **antes** que una instrucción más vieja y lenta que escribe en el mismo registro destino, dejando en el registro el valor antiguo en lugar del nuevo.

- **Regla de resolución (transparencia 11):** si dos instrucciones escriben en el mismo registro, debe asegurarse que la más vieja no escriba después que la más joven. Para lograrlo en este *pipeline* en orden, la instrucción más joven **espera en `ID` los ciclos necesarios para retrasar** su escritura en `WB` hasta quedar al menos 1 ciclo después de la más vieja.

*(Nota sobre la convención en los diagramas de la Parte I: cuando una instrucción sufre $k$ ciclos de parada en `ID`, se dibuja `IF`, seguido de $k$ marcas `id` en minúsculas, y finalmente `ID` en mayúsculas justo en el ciclo previo a entrar al operador; la instrucción inmediatamente posterior queda retenida $k$ ciclos en `if` antes de su `IF` efectivo).*

---

## 3. Ejecución fuera de orden

### 3.1. El problema de las prestaciones en orden y la idea original de Tomasulo

En el *pipeline* multiciclo en orden que acabamos de estudiar aparece un grave **problema de prestaciones** (transparencia 16):

- Pueden acumularse muchos ciclos de parada en la etapa `ID` (por dependencias RAW, riesgos estructurales con $\text{IR} < 1$, colisiones en `WB` o riesgos WAW).
- Como `ID` queda bloqueada, debe detenerse la etapa de búsqueda (`IF`) y no entran más instrucciones nuevas: **bloqueo del procesador**.
- **Causa raíz:** se fuerza a que las instrucciones se ejecuten **en el orden del programa**. Si una instrucción tiene una dependencia, espera en `ID` e impide que otras instrucciones posteriores —potencialmente independientes y listas para ejecutarse— entren en el *pipeline*, provocando además un **bajo uso de los operadores**.

**Objetivo:** desarrollar una microarquitectura que soporte **ejecución fuera de orden (*out-of-order execution*)**, como la empleada en procesadores de altas prestaciones y en los procesadores de teléfonos o tabletas actuales (la transparencia 16 cita como referencia la documentación del procesador *ARM Cortex-A75*).

#### Definiciones e idea original (transparencias 17–19)

- **Ejecución en orden (*in-order execution*):** las instrucciones se ejecutan siguiendo estrictamente el orden del programa.
- **Ejecución fuera de orden (*out-of-order* o *o-o-o*):** se permite que las instrucciones puedan ejecutarse en un orden distinto al del programa.
- **Idea original:** propuesta por **Robert Tomasulo** en IBM en **1967** para mejorar el aprovechamiento de los operadores de coma flotante de alta latencia. Para resolver el problema del bloqueo en `ID`, se añade una nueva etapa llamada **Dispatch** que vacía `ID` hacia colas de espera internas.

#### Condiciones para ejecutar una instrucción fuera de orden (transparencia 19)

Una instrucción puede pasar a ejecutarse en un operador en cuanto se cumplan **ambas condiciones**, aunque existan instrucciones más viejas pendientes en el orden del programa:

1. **Operandos fuente disponibles:** todas las instrucciones productoras anteriores de las que depende han calculado ya su resultado.
2. **Operador libre:** la unidad funcional correspondiente está disponible para aceptar la operación.

---

### 3.2. Inconvenientes, ejecución especulativa y necesidad de mantener el orden

#### El problema de las excepciones precisas (transparencia 20)

Permitir sin más que las instrucciones completen y modifiquen los registros fuera de orden conlleva un inconveniente crítico: **dificulta el soporte a excepciones precisas** por parte del Sistema Operativo (S.O.), indispensables en todos los sistemas operativos modernos.

*¿Qué sucede si una instrucción más vieja provoca una excepción (por ejemplo, un fallo de página o división por cero) cuando instrucciones más jóvenes ya han terminado y han modificado los registros?* En ese caso, el banco de registros ya no reflejaría el estado exacto producido hasta la instrucción que falló (podría contener valores sobrescritos por instrucciones posteriores que no deberían haberse ejecutado aún).

#### Ejecución especulativa (transparencias 21–22)

Además de las excepciones, los procesadores de altas prestaciones realizan **ejecución especulativa**: *las instrucciones se ejecutan y realizan `WB` «sin saber con certeza que deben ejecutarse»*.

**Escenario típico (saltos condicionales):**
1. Cuando se busca una instrucción de salto en `IF`, aún no se conoce su resultado; el procesador **predice un camino** (tomado o no tomado) y comienza a buscar instrucciones de dicho camino.
2. Con **ejecución no especulativa**, si esas instrucciones posteriores avanzan, se cancelan al resolverse el salto y **no llegan a la etapa `WB`**.
3. Con **ejecución especulativa**, dichas instrucciones **sí pueden ejecutarse y hacer `WB`**, aprovechando los operadores sin esperar al salto.

**Requisitos de la ejecución especulativa (transparencia 22):**
Para que especular sea seguro, las instrucciones especulativas pueden ejecutarse y hacer `WB`, pero **no pueden modificar el estado arquitectónico visible de la máquina**:
- Ni el **banco de registros**.
- Ni la **caché / memoria**.

En su lugar, **dejan los datos producidos en un «almacenamiento temporal»** hasta que se resuelve la especulación:
- **Si hay acierto:** las instrucciones ejecutadas se convierten en **no especulativas** y ya pueden volcar sus resultados y modificar el estado de la máquina.
- **Si hay fallo:** los datos obtenidos por todas las instrucciones ejecutadas tras *«la instrucción de salto mal especulado»* **se descartan**.

#### Necesidad de mantener el orden: la etapa *Commit* (transparencia 23)

Cuando finalmente se resuelve un salto o se detecta una excepción, *¿cómo sabemos qué instrucciones estaban antes en el programa y cuáles estaban después para cancelarlas selectivamente?*

Hace falta una estructura que **mantenga el orden original del programa** y garantice que las instrucciones **abandonen el *pipeline* en orden**. Para ello se introduce la etapa final **Commit**, único punto donde se actualiza definitivamente el estado de la máquina. De este modo se resuelven simultáneamente el soporte a **excepciones precisas** y la **ejecución especulativa**.

---

### 3.3. Etapas y estructuras de la microarquitectura o-o-o

El procesador fuera de orden con ejecución especulativa pasa a organizarse en **7 etapas** y añade **2 nuevas estructuras hardware** (transparencia 24):

| Etapas del procesador o-o-o | Nuevas estructuras |
| :--- | :--- |
| 1. `IF` (*Instruction Fetch*) | **Cola de instrucciones (*I Queue*, IQ):** almacena instrucciones decodificadas a la espera de operandos y operador |
| 2. `ID` (*Instruction Decode*) | **Reorder Buffer (ROB):** cola circular FIFO que conserva el orden del programa y actúa como almacenamiento temporal |
| 3. `Dispatch` (`Ds`) | |
| 4. `Issue` (`I`) | |
| 5. En el operador (`EX`, `A1..`, `M1..`, `D1..`, etc.) | |
| 6. `WB` (*Writeback*) | |
| 7. `Commit` (`C`) | |

#### Diagrama de la organización del procesador (transparencias 25 y 31)

```text
                  tail                                        head
                    v                                           v
                  +-----------------------------------------------+
         +------->|             Reorder Buffer (ROB)              |---+
         |        +-----------------------------------------------+   |
         |                        ^       ^       ^                   | commit
         |                        |       |       |                   v
         |                     +--+-------+-------+--+             +------+
         |                     |  Operadores (ALU,   |             |Banco |
[IF][ID][Ds] ---> [ I Queue ] ---> (Issue) -> Mult,  |             |Regis.|
                    ^ ^ ^      |   Sumador, etc.)    |             +------+
                    | | |      +----------+----------+
                    | | |                 | (Writeback)
         ===========+=+=+=================v============  <-- Common Data Bus (CDB)
```

*(Descripción visual de las transparencias 25 y 31: tras las tres etapas iniciales en orden `IF | ID | Ds`, la etapa `Dispatch` (`Ds`) envía simultáneamente la instrucción a dos sitios: a la `I Queue` y al `tail` del `ROB`. Desde la `I Queue`, la lógica de `Issue` —representada por un óvalo— selecciona instrucciones listas y las envía a los distintos operadores funcionales. Al terminar un operador, realiza `Writeback` difundiendo su resultado por el grueso bus común llamado **Common Data Bus (CDB)**: hacia arriba al `ROB` para guardar el valor temporal y marcar la instrucción como finalizada, y hacia abajo y atrás a la `I Queue` para despertar a las instrucciones dependientes que esperan ese operando. Finalmente, la etapa `commit` lee del `head` del `ROB` y escribe en el `Banco de Registros` en orden de programa).*

#### Funcionamiento detallado de cada etapa (transparencias 26–30)

1. **`IF` e `ID` (transparencia 26):**
   - `IF` busca las instrucciones en memoria. Si la instrucción es de salto, realiza aquí la **predicción** y apuesta por un camino para seguir buscando instrucciones sin esperar.
   - `ID` decodifica las instrucciones.
2. **`Dispatch` (`Ds`) (transparencia 27):** realiza **dos tareas** en orden de programa:
   - **Preparación de entrada en la *I Queue* (IQ):** envía la instrucción decodificada a la cola de instrucciones para que espere allí si le faltan operandos o el operador está ocupado; así **no bloquea la etapa `ID`**.
   - **Preparación de entrada en el ROB:** reserva una entrada en el *Reorder Buffer* para dar soporte a la ejecución especulativa y excepciones precisas. El ROB se implementa como una **cola FIFO circular** con dos punteros:
     - **`head`:** apunta a la instrucción **más vieja** (la próxima en hacer *commit*).
     - **`tail`:** apunta al extremo donde se inserta la instrucción **más joven** que acaba de llegar de `Dispatch`.
   *(Nota importante: como cada instrucción reserva su propia entrada en el ROB y los resultados temporales se asocian a esa entrada hasta el commit, los falsos conflictos **WAW** y **WAR** sobre nombres de registros no bloquean la ejecución).*
3. **`Issue` (`I`) (transparencia 28):**
   - Las instrucciones esperan en la *I Queue* hasta que tienen **todos sus operandos fuente disponibles**.
   - Una instrucción puede salir de la cola hacia ejecución cuando está **lista (*ready*)** —tiene sus operandos listos— **y su operador está libre**.
   - La lógica de `Issue` selecciona entre todas las instrucciones listas cuáles son elegidas en ese ciclo, escogiendo como máximo **una por cada operador**.
4. **`Writeback` (`WB`) (transparencia 29):**
   - El valor producido (todavía **especulativo**) se difunde por el *Common Data Bus* hacia dos destinos:
     1. **Al ROB:** se guarda en la entrada correspondiente como almacenamiento temporal y se marca que la instrucción ha **finalizado su ejecución**.
     2. **A la *I Queue*:** se entrega a todas las entradas de la cola donde haya instrucciones dependientes esperando ese valor, pasándolas a estado *ready* si ya disponen de todos sus operandos.
5. **`Commit` (`C`) (transparencia 30):**
   - Como las instrucciones entraron en orden en el ROB durante `Dispatch`, la más vieja siempre está en el `head`.
   - Cuando la instrucción situada en el **`head` del ROB** ha finalizado su ejecución:
     - **Si hay acierto de especulación** (predicción de salto correcta, o no hay excepción): escribe su resultado temporal del ROB en el **banco de registros** (haciéndolo definitivo), libera su entrada y actualiza el puntero `head` avanzando a la siguiente instrucción.
     - **Si hay fallo de especulación** (salto mal predicho): **se vacía el ROB y el resto de estructuras** (`I Queue`, etc.) con todas las instrucciones posteriores al salto erróneo, y se libera la entrada en el ROB.

---

### 3.4. Instrucciones «especiales»: accesos a memoria (*loads* y *stores*)

Las instrucciones de acceso a memoria (`load` y `store`) requieren un tratamiento especial porque involucran tanto un cálculo aritmético como el acceso a la memoria caché de datos (transparencia 34).

#### Ejecución en dos pasos y nuevas estructuras (transparencias 34–35)

Una `load` o `store` se ejecuta en **2 pasos**:

1. **Cálculo de la dirección (`AC`, *Address Calculation*):** se utiliza un **sumador específico** para sumar el registro base y el desplazamiento (por ejemplo, `0 + t1` en `fld f4, 0(t1)`).
2. **Acceso a memoria:**
   - Las **`load`**, en principio, pueden acceder a la caché en cuanto tienen calculada su dirección (etapas `L1`, `L2`) *(siempre que no haya conflicto de dirección con un `store` previo pendiente o restricción estructural en la caché)*.
   - Las **`store`** necesitan, además de haber calculado la dirección, **que esté disponible el dato** que deben escribir en memoria (etapas `S1`, `S2`).

Para coordinar estos accesos manteniendo el orden respecto a la memoria y permitir la especulación, se añade una cola específica: la **LSQ (*Load/Store Queue*)**.

- **Reserva de recursos:** cada instrucción de memoria reserva **una entrada en la LSQ** y **una entrada en el ROB**.
  *(Nota sobre las transparencias 34 y 35: el texto de la transparencia 34 dice literalmente «En el issue se reserva: Una entrada en la LSQ / Una entrada en el ROB». Sin embargo, en el diagrama de la transparencia 35 las flechas que reservan entrada en el `ROB`, en la `I Queue` y en la `LSQ` salen de la etapa **`Dispatch` (`Ds`)** —en orden de programa—, mientras que `Issue` envía las instrucciones hacia los operadores. Conceptualmente, la reserva ordenada en ROB y LSQ se realiza en `Dispatch`).*

- **Secuencia de etapas (transparencia 34):**
  ```text
  Load:   IF  ID  Ds  ...  I  AC  ...  L1  L2  WB   (y finalmente Commit C en orden)
  Store:  IF  ID  Ds  ...  I  AC  ...  S1  S2       (la escritura definitiva a caché se confirma en Commit)
  ```
  donde los puntos suspensivos `...` antes de `I` representan la posible espera en la `I Queue` (por el registro base o el sumador `AC`), y los `...` entre `AC` y `L1`/`S1` representan la espera en la `LSQ` (por disponibilidad de la caché de datos o, en el caso de una `store`, por el dato a almacenar o por su commit).

#### Diagrama del procesador o-o-o con *loads/stores* (transparencia 35)

```text
                  tail                                        head
                    v                                           v
                  +-----------------------------------------------+
         +------->|             Reorder Buffer (ROB)              |---+
         |        +-----------------------------------------------+   |
         |                        ^       ^         ^                 | commit
         |                        |       |         |                 v
         |                     +--+-------+--+   +--+--------+     +------+
         |                     | Operadores  |   |Data Cache |     |Banco |
[IF][ID][Ds] ---> [ I Queue ] ---> (Issue)   |   | (L1, L2 / |     |Regis.|
         |          ^ ^ ^      +------+------+   |  S1, S2)  |     +------+
         |          | | |             |          +-----+-----+
         +----------+ | +---->[ LSQ ] -> [ AC ] -------+
                    | | |             |                | (Writeback)
         ===========+=+=+=============+================v=  <-- Common Data Bus (CDB)
```

*(Descripción visual de la transparencia 35: respecto al diagrama de las transparencias 25 y 31, la salida de `Dispatch` (`Ds`) incorpora una tercera bifurcación inferior hacia la **LSQ**, que alimenta al sumador dedicado **AC** y este a la **Data Cache** —dibujada en verde—. La salida de la caché de datos conecta con el **Common Data Bus (CDB)** para realizar `Writeback` hacia el `ROB` y hacia la `I Queue` en las operaciones `load`.)*

---

### 3.5. Nota sobre el tratamiento de los saltos

El índice interno de la sección *Ejecución fuera de orden* en la transparencia 17 (`Procesadores avanzados 1a y 2a parte.pdf`) incluye un quinto punto titulado `5. Saltos`. En las transparencias facilitadas en los PDF, los saltos se cubren dentro de las etapas explicadas en los apartados anteriores:
- En **`IF`** (transparencia 26): al buscar una instrucción de salto se realiza una predicción y se continúa buscando instrucciones especulativas del camino predicho.
- En **`Commit`** (transparencia 30): cuando el salto alcanza el `head` del ROB y se comprueba su resultado real, si hubo acierto las instrucciones posteriores continúan su *commit* normal; si hubo fallo de especulación se vacían el ROB y el resto de estructuras (`I Queue`, `LSQ`) para descartar el camino erróneo y reanudar la búsqueda en la dirección correcta.

El estudio detallado de los predictores hardware corresponde al bloque 3 del índice general (*Predicción dinámica*), no incluido en estos PDF.

---

# Parte II: Resolución paso a paso de las actividades

> **Nota sobre el material de actividades:** no se proporcionó un PDF de boletín de actividades separado para este tema. Esta Parte II reúne y resuelve paso a paso **todas** las cuestiones y ejercicios propuestos a lo largo de las transparencias de `Procesadores avanzados 1a parte.pdf` y `Procesadores avanzados 1a y 2a parte.pdf`, respetando su orden y numeración de página.

## Actividades 1: Cuestiones y ejercicios de operaciones multiciclo

### Cuestión: Operador segmentado vs. no segmentado, latencia e *Initiation Rate* (p. 5)

*Enunciado (transparencia 5): Se dispone de un multiplicador de coma flotante de $4\text{ ns}$ en un procesador que trabaja a $1\text{ GHz}$ ($T_{\text{reloj}} = 1\text{ ns}$).*
1. *Si se diseña segmentado en 4 etapas (`M1`, `M2`, `M3`, `M4`), ¿cuáles serían los registros de segmentación? ¿Durante cuántos ciclos ocuparía el operador una multiplicación? Si en el ciclo 1 empieza una multiplicación, ¿en qué ciclo puede empezar la siguiente?*
2. *Si se diseña no segmentado, responde a las dos preguntas anteriores.*
3. *¿Cuál sería la latencia y cuál el Initiation Rate ($\text{IR}$) en ambos casos?*

**Resolución paso a paso:**

**1. Caso segmentado en 4 etapas (`M1`, `M2`, `M3`, `M4`):**
- **Registros de segmentación:** se precisan registros intermedios entre cada pareja de etapas consecutivas del multiplicador: **`M1/M2`**, **`M2/M3`** y **`M3/M4`** (además de los registros de interfaz del cauce general: **`ID/M1`** a la entrada del operador y **`M4/WB`** a su salida).
- **Ciclos que ocupa la operación:** como $f = 1\text{ GHz}$, cada ciclo dura $T_{\text{reloj}} = 1/10^9\text{ s} = 1\text{ ns}$. Una multiplicación tarda $4\text{ ns} / 1\text{ ns} = \mathbf{4\text{ ciclos}}$ en atravesar el multiplicador (1 ciclo en `M1`, 1 en `M2`, 1 en `M3` y 1 en `M4`).
- **Inicio de la siguiente multiplicación:** como cada etapa solo retiene la instrucción $1$ ciclo, al pasar la primera multiplicación a `M2` en el ciclo $2$, la etapa `M1` queda libre. Por tanto, la siguiente multiplicación puede empezar en el **ciclo 2**.

**2. Caso no segmentado (mismo operador de $4\text{ ns}$ y reloj a $1\text{ GHz}$):**
- **Ciclos que ocupa el operador:** la multiplicación ocupa todo el circuito combinacional completo durante **4 ciclos** consecutivos (ciclos $1, 2, 3$ y $4$).
- **Inicio de la siguiente multiplicación:** hasta que la primera multiplicación no termina al final del ciclo $4$, el operador no queda libre. La siguiente multiplicación solo puede empezar en el **ciclo 5**.

**3. Latencia e *Initiation Rate* ($\text{IR}$):**
- **Latencia:** en ambos diseños, el circuito tarda $4\text{ ns} = \mathbf{4\text{ ciclos}}$ en completar la multiplicación.
- ***Initiation Rate* ($\text{IR}$):**
  - **Segmentado:** deben transcurrir $k = 1$ ciclo antes de poder iniciar una nueva operación $\implies \mathbf{\text{IR} = \dfrac{1}{1} = 1}$.
  - **No segmentado:** deben transcurrir $k = 4$ ciclos antes de poder iniciar una nueva operación $\implies \mathbf{\text{IR} = \dfrac{1}{4} = 0.25}$.

---

### Problema: Ejercicio multiciclo 1 (p. 12)

*Enunciado (transparencia 12): Realiza el diagrama de ejecución en el pipeline multiciclo en orden con cortocircuitos para el siguiente fragmento de código, sabiendo que:*
- *Multiplicador c.f.: latencia $4$ (`M1`, `M2`, `M3`, `M4`), $\text{IR} = 1$.*
- *Sumador c.f.: latencia $2$ (`A1`, `A2`), $\text{IR} = 1$.*

```text
fmul.d f0,f0,f1
fadd.d f2,f2,f0
fadd.d f3,f2,f4
fmul.d f0,f2,f0
```

**Resolución paso a paso:**

*(Aclaración sobre el modelo aplicado: seguimos exactamente las reglas y notación de las transparencias 9, 10, 11 y 15: ejecución en orden en `IF` e `ID`, lectura/comprobación de operandos en `ID`, cortocircuito desde la etapa de salida/WB hacia `A1`/`M1`, y escritura/lectura simultánea en el banco de registros en el mismo ciclo de `WB` como en la transparencia 15).*

**Paso 1: Identificación de dependencias y riesgos**
1. `I1: fmul.d f0,f0,f1` escribe `f0`.
2. `I2: fadd.d f2,f2,f0` lee `f0` $\implies$ **dependencia RAW** respecto a `I1`.
3. `I3: fadd.d f3,f2,f4` lee `f2` $\implies$ **dependencia RAW** respecto a `I2`.
4. `I4: fmul.d f0,f2,f0` lee `f2` (dependencia **RAW** respecto a `I2`), lee `f0` (de `I1`) y escribe `f0` (posible riesgo **WAW** respecto a `I1`, aunque `I1` habrá terminado mucho antes).
5. **Riesgos estructurales de operadores:** no hay, porque ambos operadores tienen $\text{IR} = 1$ (están segmentados).
6. **Riesgos estructurales de escritura (`WB`):** comprobaremos que ninguna instrucción coincida en el mismo ciclo de `WB`.

**Paso 2: Trazado ciclo a ciclo**
- **`I1: fmul.d f0,f0,f1`**: entra `IF` en ciclo $1$, `ID` en ciclo $2$, ejecuta `M1, M2, M3, M4` en los ciclos $3, 4, 5, 6$ y hace `WB` en el ciclo **$7$**. El nuevo valor de `f0` está disponible al terminar `M4` (final del ciclo $6$) para pasar por cortocircuito en el ciclo $7$.
- **`I2: fadd.d f2,f2,f0`**: hace `IF` en ciclo $2$. En los ciclos $3, 4, 5$ se detiene en `ID` (`id id id`) porque `I1` aún está en `M1, M2, M3`. En el ciclo $6$ realiza su `ID` definitivo, entra en `A1` en el ciclo **$7$** (recibiendo `f0` por cortocircuito desde `WB` de `I1`), pasa a `A2` en el ciclo $8$ y hace `WB` en el ciclo **$9$**.
- **`I3: fadd.d f3,f2,f4`**: entra en `if` durante los ciclos $3, 4, 5$ porque `I2` bloquea `ID`. Hace `IF` en el ciclo $6$, espera $1$ ciclo en `id` en el ciclo $7$ (porque `I2` está en `A1` y aún no ha terminado `A2`), realiza `ID` en el ciclo $8$, entra en `A1` en el ciclo **$9$** (cortocircuito desde `WB` de `I2`), pasa a `A2` en el ciclo $10$ y hace `WB` en el ciclo **$11$**.
- **`I4: fmul.d f0,f2,f0`**: queda bloqueada en `if` en el ciclo $7$ mientras `I3` retiene `ID`. Hace `IF` en el ciclo $8$ e `ID` en el ciclo $9$. En ese ciclo $9$, `I2` ya está en `WB` escribiendo `f2` (y `f0` ya está en el banco desde el ciclo $7$), por lo que `I4` no necesita detenerse y entra en `M1, M2, M3, M4` en los ciclos $10, 11, 12, 13$, finalizando con `WB` en el ciclo **$14$**.
- **Comprobación de `WB` y WAW:** los ciclos de `WB` son $7$ (`I1`), $9$ (`I2`), $11$ (`I3`) y $14$ (`I4`). Todos son distintos (sin colisión en el banco) y $14 > 7$ (sin riesgo WAW sobre `f0`).

**Diagrama final:**

```text
                  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15
fmul.d f0,f0,f1  IF ID M1 M2 M3 M4 WB
fadd.d f2,f2,f0     IF id id id ID A1 A2 WB
fadd.d f3,f2,f4        if if if IF id ID A1 A2 WB
fmul.d f0,f2,f0                    if IF ID M1 M2 M3 M4 WB
```

**Conclusión:** la secuencia completa termina su etapa `WB` en el **ciclo 14**. *(Nota: en la cabecera de la transparencia 12 figura `19` por errata tipográfica en lugar de `10`).*

---

### Problema: Ejercicio multiciclo 2 (p. 14-15)

*Enunciado (transparencia 14): Realiza el diagrama en el pipeline multiciclo en orden para la siguiente secuencia, con:*
- *Sumador c.f.: latencia $2$ (`A1`, `A2`), $\text{IR} = 1$.*
- *Multiplicador c.f.: latencia $3$ (`M1`, `M2`, `M3`), $\text{IR} = 1$.*
- *Divisor c.f.: latencia $4$ (`D1`, `D2`, `D3`, `D4`), $\text{IR} = 1/4$. (Nota: la transparencia 14 escribe por errata `D1, D2, D3, D2`).*
- *Loads: ruta de enteros, etapa `M` $= 1$ ciclo (`IF ID EX M WB`).*

```text
fld f1, 0(x1)
fdiv.d f4,f0,f1
fadd.d f2,f3,f4
fld f4, 8(x1)
fmul.d f3,f4,f2
fld f5, 16(x1)
```

**Resolución paso a paso (verificación de la solución de la transparencia 15):**

Este ejercicio aparece resuelto en la **transparencia 15** de `Procesadores avanzados 1a y 2a parte.pdf` (no incluida en `1a parte.pdf`). A continuación justificamos cada línea del diagrama oficial:

1. **`fld f1, 0(x1)`:** recorre `IF`(1) $\to$ `ID`(2) $\to$ `EX`(3) $\to$ `M`(4) $\to$ `WB`(5). El dato `f1` sale de caché al final del ciclo $4$ (`M`).
2. **`fdiv.d f4,f0,f1`:** depende **RAW** de `f1`. Hace `IF`(2), espera $1$ ciclo en `id`(3) mientras `fld` está en `EX`, hace `ID`(4), entra al divisor en el ciclo $5$ (cortocircuito desde `WB` de `fld` en el ciclo $5$ a `D1`), ejecuta `D1, D2, D3, D4` en los ciclos $5, 6, 7, 8$ y hace `WB` en el ciclo **$9$**.
   *(Nota sobre la transparencia 15: en el diagrama de la diapositiva se escribe por errata tipográfica `M1 M2 M3 M4` en lugar de `D1 D2 D3 D4` para `fdiv.d`).*
3. **`fadd.d f2,f3,f4`:** depende **RAW** de `f4` producido por `fdiv.d`. Bloqueada en `if`(3) por `fdiv.d`, hace `IF`(4), espera $3$ ciclos en `id`(5, 6, 7) hasta que `fdiv.d` termine `D4` en el ciclo $8$, hace `ID`(8), entra en `A1`(9) con cortocircuito desde `WB` de `fdiv.d`, `A2`(10) y `WB`(11).
4. **`fld f4, 8(x1)`:** tiene un posible riesgo **WAW** sobre `f4` respecto a `fdiv.d` (`WB` en ciclo $9$). Sin embargo, al quedar retenida en `if if if` (ciclos $5, 6, 7$) mientras `fadd.d` espera en `ID`, no alcanza `IF` hasta el ciclo $8$, `ID`(9), `EX`(10), `M`(11) y `WB`(12). Como $12 > 9$, el riesgo WAW ya está superado sin añadir paradas adicionales.
5. **`fmul.d f3,f4,f2`:** depende **RAW** de `f4` (producido por `fld f4, 8(x1)`, disponible tras `M` en ciclo $11$) y de `f2` (producido por `fadd.d`, disponible tras `A2` en ciclo $10$). Hace `IF`(9), espera $1$ ciclo en `id`(10) esperando a `f4`, hace `ID`(11), entra en `M1`(12) con cortocircuito de `WB` de `fld`, `M2`(13), `M3`(14) y `WB`(15).
6. **`fld f5, 16(x1)`:** no tiene dependencias de datos ni riesgo WAW (escribe en `f5`), ni riesgo estructural en el operador entero. Queda en `if`(10) mientras `fmul.d` está en `id`, y hace `IF`(11).
   **¿Por qué espera `1` ciclo en `id`(12)?** Si hiciera `ID`(12), seguiría `EX`(13) $\to$ `M`(14) $\to$ `WB`(15), ¡y **colisionaría en `WB` en el ciclo 15** con `fmul.d f3,f4,f2`, pues ambas escriben en el banco de registros de coma flotante! Aplicando la regla de la transparencia 11 (*riesgo estructural por escritura en el banco*), la instrucción más joven (`fld f5`) espera **1 ciclo en `ID`** (`id` en ciclo $12$, `ID` en $13$) para desplazar su `WB` al **ciclo 16**.

**Diagrama completo (transparencia 15, con `D1..D4` corregido):**

```text
                  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16
fld f1, 0(x1)    IF ID EX  M WB
fdiv.d f4,f0,f1     IF id ID D1 D2 D3 D4 WB
fadd.d f2,f3,f4        if IF id id id ID A1 A2 WB
fld f4, 8(x1)                 if if if IF ID EX  M WB
fmul.d f3,f4,f2                           IF id ID M1 M2 M3 WB
fld f5, 16(x1)                               if IF id ID EX  M WB
```

**Conclusión:** el bloque finaliza en el **ciclo 16**.

---

## Actividades 2: Ejercicios de ejecución fuera de orden (o-o-o)

*(Nota metodológica general sobre los diagramas o-o-o de las transparencias 32, 34, 36 y 38: en el procesador fuera de orden escalar de este tema, las instrucciones entran en orden por `IF`, `ID` y `Ds` —1 ciclo por etapa— sin bloquear `ID`. Cuando pasan a `I Queue` / `LSQ`, la etapa **`I` (*Issue*)** ocupa 1 ciclo previo a la entrada al operador —como muestra explícitamente la transparencia 34: `IF ID Ds ... I AC ... L1 L2 WB`— y puede ejecutarse en el mismo ciclo en que la instrucción productora difunde su dato por el CDB en `WB`, o en el último ciclo de un operador no segmentado para entrar al ciclo siguiente cuando quede libre. Al finalizar la ejecución se realiza **`WB`** en el CDB y finalmente **`C` (*Commit*)** en el `head` del ROB, al menos 1 ciclo después de `WB` y en estricto orden de programa, a razón de 1 commit por ciclo.)*

### Problema: Ejercicio o-o-o 1 (p. 32)

*Enunciado (transparencia 32): Dadas las siguientes características:*
- *Multiplicador/divisor: $3$ ciclos (`M1`, `M2`, `M3`), $\text{IR} = 1/3$ (no segmentado).*
- *Sumador/restador: $2$ ciclos (`A1`, `A2`), $\text{IR} = 1$ (segmentado).*

*Realiza el diagrama de ejecución fuera de orden para:*

```text
fmul.d f0,f1,f2
fmul.d f1,f0,f2
fadd.d f2,f0,f3
fmul.d f3,f4,f5
```

**Resolución paso a paso:**

**Paso 1: Análisis de dependencias verdaderas y recursos**
- `I1: fmul.d f0,f1,f2`: sin dependencias. Produce `f0`.
- `I2: fmul.d f1,f0,f2`: **RAW `f0`** con `I1`. Además necesita el multiplicador ($\text{IR} = 1/3$). (Nota: la escritura de `I2` en `f1` no genera riesgo WAR frente a `I1`, porque `I1` ya leyó `f1` al emitirse y cada resultado se guarda en el ROB hasta el *commit* en orden).
- `I3: fadd.d f2,f0,f3`: **RAW `f0`** con `I1`. Usa el sumador (`A1, A2`), distinto del multiplicador.
- `I4: fmul.d f3,f4,f5`: **sin dependencias** respecto a `I1`, `I2` ni `I3` (los registros `f4, f5` están disponibles desde el inicio y el posible riesgo WAR sobre `f3` con `I3` queda resuelto por el ROB). Solo compite por el **multiplicador** ($\text{IR} = 1/3$).

**Paso 2: Traza ciclo a ciclo y ejecución fuera de orden**
1. **`I1: fmul.d f0,f1,f2`:** `IF`(1) $\to$ `ID`(2) $\to$ `Ds`(3) $\to$ `I`(4) $\to$ `M1`(5) $\to$ `M2`(6) $\to$ `M3`(7) $\to$ `WB`(8) $\to$ `C`(9). Ocupa el multiplicador en los ciclos $5, 6$ y $7$; difunde `f0` por el CDB en el **ciclo $8$**.
2. **`I2: fmul.d f1,f0,f2`:** entra en orden `IF`(2) $\to$ `ID`(3) $\to$ `Ds`(4) y queda en la *I Queue* esperando a `f0`. En el ciclo $8$, cuando `I1` hace `WB`, `I2` recibe `f0` y queda lista. **Sin embargo**, ¡el multiplicador acaba de ser asignado a `I4` para los ciclos $8, 9, 10$! Por tanto `I2` debe esperar en la *I Queue* hasta hacer `I`(10) $\to$ `M1`(11) $\to$ `M2`(12) $\to$ `M3`(13) $\to$ `WB`(14) $\to$ `C`(15).
3. **`I3: fadd.d f2,f0,f3`:** entra en orden `IF`(3) $\to$ `ID`(4) $\to$ `Ds`(5) y espera en la *I Queue* el valor `f0` de `I1`. Lo recibe en el ciclo $8$ (`WB` de `I1`), hace `I`(8) $\to$ `A1`(9) $\to$ `A2`(10) $\to$ `WB`(11). Aunque termina en el ciclo $11$, debe esperar en el ROB a que `I2` haga *commit* en el ciclo $15$, por lo que su *commit* es en el **ciclo 16**.
4. **`I4: fmul.d f3,f4,f5`:** entra en orden `IF`(4) $\to$ `ID`(5) $\to$ `Ds`(6). Ya tiene `f4` y `f5` disponibles desde que llega a la *I Queue* en el ciclo $6$, mientras `I2` e `I3` siguen bloqueadas esperando `f0`. En cuanto `I1` está en su último ciclo de multiplicación (`M3` en el ciclo $7$), `I4` hace `I`(7) y **se ejecuta fuera de orden** antes que `I2` e `I3`: `M1`(8) $\to$ `M2`(9) $\to$ `M3`(10).
   - *Uso del CDB en `WB`:* en el ciclo $10$ terminan a la vez `I3` (`A2`) e `I4` (`M3`). Con un único *Common Data Bus* (transparencia 25), `I3` (más vieja) toma `WB` en el ciclo $11$ e `I4` (más joven) hace `WB` en el ciclo $12$ *(si hubiera puertos independientes en el bus para cada operador, `I4` haría `WB` en el 11; en ambos casos el commit no cambia)*.
   - Finalmente, `I4` espera en el ROB hasta que hagan *commit* `I1`(9), `I2`(15) e `I3`(16), realizando su `C` en el **ciclo 17**.

**Diagrama final:**

```text
                  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17
fmul.d f0,f1,f2  IF ID Ds  I M1 M2 M3 WB  C
fmul.d f1,f0,f2     IF ID Ds           I M1 M2 M3 WB  C
fadd.d f2,f0,f3        IF ID Ds     I A1 A2 WB        C
fmul.d f3,f4,f5           IF ID Ds  I M1 M2 M3    WB     C
```

**Conclusión:** gracias a la ejecución fuera de orden, `I4` aprovecha el multiplicador entre los ciclos $8$ y $10$ mientras `I2` esperaba a `I1`, evitando dejar ocioso el operador y completando el *commit* de las 4 instrucciones en el **ciclo 17**.

---

### Problema: Ejercicio o-o-o con *loads* (p. 36)

*Enunciado (transparencia 36): Dadas las siguientes latencias:*
- *Suma c.f.: $3$ ciclos (`A1`, `A2`, `A3`), $\text{IR} = 1$.*
- *Multiplicación c.f.: $4$ ciclos (`M1`, `M2`, `M3`, `M4`), $\text{IR} = 1$. (Nota: la transparencia 36 escribe por errata `4 ciclos (A1, A2, A3)`).*
- *Load: $2$ ciclos (`L1`, `L2`), $\text{IR} = 1$ (precedidos del cálculo de dirección `AC` de 1 ciclo según la transparencia 34).*

*Realiza el diagrama para:*

```text
fadd.d f4,f2,f0
fmul.d f6,f4,f0
fld f4,0(t1)
fmul.d f6,f6,f4
```

*(Nota sobre la transparencia 36: en el enunciado aparece una letra `C` suelta al final de `fld f4,0(t1) C`, probablemente un error tipográfico de edición en la diapositiva).*

**Resolución paso a paso:**

**Paso 1: Análisis de dependencias**
- `I1: fadd.d f4,f2,f0`: sin dependencias. Produce `f4`.
- `I2: fmul.d f6,f4,f0`: **RAW `f4`** con `I1`. Produce `f6`.
- `I3: fld f4, 0(t1)`: escribe `f4`. En un procesador en orden tendría riesgo **WAW** con `I1` y riesgo **WAR** con `I2` (sobre `f4`). En el procesador **o-o-o con ROB**, estas dependencias falsas por nombre se eliminan: `I2` leerá el `f4` producido por `I1`, mientras que `I3` escribirá el nuevo `f4` en su propia entrada del ROB sin interferir con `I1` ni con `I2` y hará *commit* después de `I2`.
- `I4: fmul.d f6,f6,f4`: **RAW `f6`** con `I2` y **RAW `f4`** con `I3` (última productora de `f4` en orden de programa).

**Paso 2: Traza ciclo a ciclo**
1. **`I1: fadd.d f4,f2,f0`:** `IF`(1) $\to$ `ID`(2) $\to$ `Ds`(3) $\to$ `I`(4) $\to$ `A1`(5) $\to$ `A2`(6) $\to$ `A3`(7) $\to$ `WB`(8) $\to$ `C`(9).
2. **`I2: fmul.d f6,f4,f0`:** `IF`(2) $\to$ `ID`(3) $\to$ `Ds`(4). Espera en la *I Queue* el resultado `f4` de `I1`, que llega por el CDB en el ciclo $8$. Hace `I`(8) $\to$ `M1`(9) $\to$ `M2`(10) $\to$ `M3`(11) $\to$ `M4`(12) $\to$ `WB`(13) $\to$ `C`(14).
3. **`I3: fld f4, 0(t1)`:** `IF`(3) $\to$ `ID`(4) $\to$ `Ds`(5). Como `t1` está disponible y el sumador `AC` y la caché están libres, se ejecuta **fuera de orden** sin esperar a `I1` ni a `I2`: `I`(6) $\to$ `AC`(7) $\to$ `L1`(8) $\to$ `L2`(9) $\to$ `WB`(10). Como `I2` no hace *commit* hasta el ciclo $14$, `I3` retiene su valor en el ROB y hace `C` en el **ciclo 15**.
4. **`I4: fmul.d f6,f6,f4`:** `IF`(4) $\to$ `ID`(5) $\to$ `Ds`(6). Necesita `f4` (disponible ya en el ciclo $10$ gracias a `I3`) y `f6` (que no llega hasta el `WB` de `I2` en el ciclo $13$). Por tanto, hace `I`(13) $\to$ `M1`(14) $\to$ `M2`(15) $\to$ `M3`(16) $\to$ `M4`(17) $\to$ `WB`(18) $\to$ `C`(19).

**Diagrama final:**

```text
                  01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19
fadd.d f4,f2,f0   IF ID Ds  I A1 A2 A3 WB  C
fmul.d f6,f4,f0      IF ID Ds           I M1 M2 M3 M4 WB  C
fld f4, 0(t1)           IF ID Ds  I AC L1 L2 WB           C
fmul.d f6,f6,f4            IF ID Ds                    I M1 M2 M3 M4 WB  C
```

**Conclusión:** el programa completa todas sus instrucciones en el **ciclo 19** (coincidiendo exactamente con las 19 columnas de la plantilla de la transparencia 36). Nótese cómo `fld f4, 0(t1)` se adelanta ejecutándose entre los ciclos $6$ y $10$ sin corromper el registro `f4` que todavía necesitaba `fmul.d f6,f4,f0`, gracias al almacenamiento temporal en el ROB.

---

### Problema: Ejercicio o-o-o 2 (p. 38)

*Enunciado (transparencia 38): Dadas las siguientes características:*
- *Unidad de carga: $2$ ciclos (`L1`, `L2`), $\text{IR} = 1/2$ (precedidos de `AC` de 1 ciclo).*
- *Sumador/restador: $2$ ciclos (`A1`, `A2`), $\text{IR} = 1$.*
- *Multiplicador/divisor: `4` ciclos (`M1`, `M2`, `M3`, `M4`), $\text{IR} = 1$.*
  *(Nota importante sobre la transparencia 38: la diapositiva dice literalmente «Mult/div 7 ciclos (M1, M2, M3, M4), IR = 1», lo cual contiene una contradicción entre «7 ciclos» y las 4 etapas enumeradas `M1, M2, M3, M4`. Resolvemos el diagrama completo con las 4 etapas `M1..M4` indicadas entre paréntesis, y al final anotamos el desplazamiento exacto si se tomara latencia 7).*

*Realiza el diagrama para:*

```text
fld f1, 0(t1)
fld f2, b(t2)
fmul.d f0,f2,f4
fsub.d f3,f2,f1
fdiv.d f5,f0,f1
fadd.d f0,f3,f2
```

**Resolución paso a paso:**

**Paso 1: Análisis de dependencias y riesgos**
- `I1: fld f1, 0(t1)`: sin dependencias. Produce `f1`.
- `I2: fld f2, b(t2)`: sin dependencias de datos con `I1`, pero la unidad de carga (`L1, L2`) tiene **$\text{IR} = 1/2$** (no segmentada), por lo que no puede entrar a `L1` hasta que `I1` libere `L2`. Produce `f2`.
- `I3: fmul.d f0,f2,f4`: **RAW `f2`** con `I2`. Produce `f0`.
- `I4: fsub.d f3,f2,f1`: **RAW `f2`** con `I2` y **RAW `f1`** con `I1`. Produce `f3`.
- `I5: fdiv.d f5,f0,f1`: **RAW `f0`** con `I3` y **RAW `f1`** con `I1`. Produce `f5`.
- `I6: fadd.d f0,f3,f2`: **RAW `f3`** con `I4` y **RAW `f2`** con `I2`. Además escribe `f0` (falsa dependencia **WAW** con `I3` y **WAR** con `I5`, que se eliminan mediante el ROB y la *I Queue*).

**Paso 2: Traza ciclo a ciclo (con `M1..M4` de 4 ciclos)**
1. **`I1: fld f1, 0(t1)`:** `IF`(1) $\to$ `ID`(2) $\to$ `Ds`(3) $\to$ `I`(4) $\to$ `AC`(5) $\to$ `L1`(6) $\to$ `L2`(7) $\to$ `WB`(8) $\to$ `C`(9).
2. **`I2: fld f2, b(t2)`:** `IF`(2) $\to$ `ID`(3) $\to$ `Ds`(4) $\to$ `I`(5) $\to$ `AC`(6). Al terminar `AC` en el ciclo $6$, la unidad de caché (`L1, L2`, con $\text{IR} = 1/2$) sigue ocupada por `I1` durante los ciclos $6$ y $7$. Por tanto, `I2` espera 1 ciclo en la `LSQ` (ciclo $7$) y entra a `L1`(8) $\to$ `L2`(9) $\to$ `WB`(10) $\to$ `C`(11). *(Nota: si el sumador `AC` de direcciones fuese compartido con `IR=1`, no hay conflicto porque `I1` lo usa en el ciclo 5 e `I2` en el 6).*
3. **`I3: fmul.d f0,f2,f4`:** `IF`(3) $\to$ `ID`(4) $\to$ `Ds`(5). Espera `f2` en la *I Queue* hasta el `WB` de `I2` (ciclo $10$). Hace `I`(10) $\to$ `M1`(11) $\to$ `M2`(12) $\to$ `M3`(13) $\to$ `M4`(14) $\to$ `WB`(15) $\to$ `C`(16).
4. **`I4: fsub.d f3,f2,f1`:** `IF`(4) $\to$ `ID`(5) $\to$ `Ds`(6). Recibe `f1` en el ciclo $8$ y `f2` en el ciclo $10$. Ya lista, hace `I`(10) $\to$ `A1`(11) $\to$ `A2`(12) $\to$ `WB`(13). Espera en el ROB tras `I3` para hacer `C` en el **ciclo 17**.
5. **`I5: fdiv.d f5,f0,f1`:** `IF`(5) $\to$ `ID`(6) $\to$ `Ds`(7). Ya tiene `f1` desde el ciclo $8$, pero necesita `f0` de `I3` (`WB` en ciclo $15$). Por ello queda esperando hasta hacer `I`(15) $\to$ `M1`(16) $\to$ `M2`(17) $\to$ `M3`(18) $\to$ `M4`(19) $\to$ `WB`(20) $\to$ `C`(21).
6. **`I6: fadd.d f0,f3,f2`:** `IF`(6) $\to$ `ID`(7) $\to$ `Ds`(8). Ya tiene `f2` desde el ciclo $10$ y recibe `f3` cuando `I4` hace `WB` en el ciclo $13$. Como no depende de `I5` (y el ROB resuelve tanto el WAW con `I3` como el WAR con `I5`), **se ejecuta fuera de orden antes que `I5`**: hace `I`(13) $\to$ `A1`(14) $\to$ `A2`(15) $\to$ `WB`(16). Permanece en el ROB hasta después del *commit* de `I5`(21), realizando su `C` en el **ciclo 22**.
- **Comprobación del CDB (`WB`):** los ciclos de `WB` son $8$ (`I1`), $10$ (`I2`), $13$ (`I4`), $15$ (`I3`), $16$ (`I6`) y $20$ (`I5`); no existe ninguna colisión en el CDB.

**Diagrama final:**

```text
                  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22
fld f1, 0(t1)    IF ID Ds  I AC L1 L2 WB  C
fld f2, b(t2)       IF ID Ds  I AC    L1 L2 WB  C
fmul.d f0,f2,f4        IF ID Ds              I M1 M2 M3 M4 WB  C
fsub.d f3,f2,f1           IF ID Ds           I A1 A2 WB        C
fdiv.d f5,f0,f1              IF ID Ds                       I M1 M2 M3 M4 WB  C
fadd.d f0,f3,f2                 IF ID Ds              I A1 A2 WB                 C
```

**Conclusión:** con latencia $4$ (`M1..M4`) para el multiplicador/divisor, el programa completa su último *commit* en el **ciclo 22**, apreciándose claramente cómo `fadd.d f0,f3,f2` adelanta en ejecución a `fdiv.d f5,f0,f1` sin causar conflictos WAW ni WAR. *(Nota: si el operador `mult/div` tuviera latencia de $7$ ciclos `M1..M7` con $\text{IR}=1$, entonces `I3` haría `M1..M7` en los ciclos $11..17$, `WB`(18) y `C`(19); `I4` haría `C`(20); `I5` haría `I`(18), `M1..M7` en $19..25$, `WB`(26) y `C`(27); e `I6` seguiría ejecutándose fuera de orden en los ciclos $14..16$ esperando hasta su `C`(28)).*